### Notebook 00: Data Collection — AI News Headline Scraping

**CS 210 Final Project** | What the Headlines Say: Measuring Cognitive Bias and Agency Framing in AI News Discourse | Sadhana Vasanthakumar

This notebook scrapes AI-related news headlines from Google News using the `gnews` library. The final dataset covers August 2024 through March 2026 — 179,372 unique headlines from 13,019 publishers across 20 monthly windows. Queries are organized into 9 topic tiers (192 queries total) to make sure coverage isn't biased toward any one framing of AI.

**Don't re-run this from scratch.** The checkpoint files from the original run are the actual reproducible artifact — Google News results change daily, so a fresh run would produce a different sample. The original session log is in `scraping_results_summary.txt` if you want to verify what ran.

#### Output files

| File | Rows | Notes |
|------|------|-------|
| `data/ai_headlines_raw.csv` | 179,372 | 4-column Figshare-compatible schema |
| `data/ai_headlines_full.csv` | 179,372 | Full metadata (10 cols), used by notebook 01 |
| `data/ai_headlines_website.csv` | 3,000 | Stratified monthly sample for the dashboard |
| `checkpoints/checkpoint_YYYY-MM.csv` | ~6–11K each | 20 checkpoint files, one per window |

### Dependencies

```bash
pip install gnews pandas numpy
```

No API key needed. Everything else is standard library.

#### How to run

The scraper took **667 minutes** when I ran it (April 21, 2026). Rate limiting is set to 2.5s per query and 8s between windows — this is intentional, removing it will get you blocked by Google News. Make sure your laptop won't sleep before starting cell 7.

1. Activate your virtual environment
2. Run cells 1–6 (under 5 seconds total — just definitions)
3. Run cell 7 and leave it running (6–8 hours). You can interrupt safely at any time and resume — completed windows are checkpointed and will be skipped on re-run
4. After cell 7 finishes, run cells 8–11 to merge, validate, and export

In [1]:
#CELL 1 - Imports and project folder setup
'''
Imports all required libraries and creates the checkpoints/
and data/ folders on the local hard drive. Both os.makedirs
calls use exist_ok=True so re-running is always safe.

Key decision -- why gnews over alternatives considered:
tried NewsAPI first but the free tier caps at 100 requests/day which
would've limited us to ~10k headlines total — not enough for 20 months
of trend analysis. 

GDELT was a dead end too; it returns event records
not headline text, and filtering out the noise took longer than it
was worth. 

gnews works directly with Google News RSS, no API key,
no hard rate limits, and returns clean title/publisher/date fields.

'''
from gnews import GNews
import pandas as pd
import numpy as np
import time
import os
import re
from datetime import datetime
from collections import Counter

# create folder structure -- os.makedirs with exist_ok=True is safe to re-run
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("data", exist_ok=True)

print(f"working directory: {os.getcwd()}")
print(f"run started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

working directory: c:\Users\Sadhana\Desktop\cs210_bias_project
run started: 2026-05-04 18:35:01


In [2]:
# CELL 2 -- Keyword architecture: 192 queries across 9 tiers

'''
Defines every search query used in the scrape, organized
into nine thematic tiers. Combines them into ALL_QUERIES
and maps each query to a topic category for the website.

Key decision -- why 192 queries instead of one broad query:
gnews caps results at 100 per query. A single query like
"artificial intelligence" returns the 100 most recent popular
articles, dominated by major outlets. Domain-specific queries
(e.g. "AI courtroom", "AI grief") surface niche publishers
and specialized framing that broad queries miss entirely.

Tier Structure: 
Tier 1 Core           -- broadest coverage, highest volume
Tier 2 Domains        -- sector-specific AI applications
Tier 3 Risk/Fear      -- feeds AAI (AI Anxiety Index) scoring
Tier 4 Agency         -- feeds PSI (Power Shift Index) scoring
Tier 5 Named Entities -- specific models/companies drive spikes
Tier 6 Policy         -- regulation, law, governance
Tier 7 Human Impact   -- key for Invisible Human Framework
Tier 8 Democracy      -- Nov 2024 US election coverage spike
Tier 9 Infrastructure -- AI energy/data center stories (2025)

'''

# tier 1: broadest terms, highest volume
TIER_1_CORE = [
    "artificial intelligence",
    "machine learning",
    "AI technology",
    "generative AI",
    "large language model",
    "AI model",
    "deep learning",
    "neural network AI",
    "agentic AI",
    "AI agents",
    "multimodal AI",
    "AI assistant",
    "foundation model",
]

# tier 2: sector-specific applications
TIER_2_DOMAINS = [
    "AI healthcare", "AI medicine", "AI drug discovery",
    "AI mental health", "AI diagnosis", "AI radiology", "AI surgery",
    "AI education", "AI schools", "AI students cheating", "AI tutoring",
    "AI finance", "AI banking", "AI stock market", "AI trading",
    "AI insurance", "AI real estate",
    "AI legal", "AI courtroom", "AI government", "AI judge",
    "AI military", "AI defense", "AI cybersecurity", "AI weapons",
    "AI facial recognition", "AI policing", "AI tracking",
    "AI climate", "AI environment", "AI agriculture",
    "AI energy", "AI wildfire",
    "AI transportation", "AI self-driving", "AI robotics",
    "AI art", "AI music", "AI writing", "AI journalism",
    "AI film", "AI video", "AI image generation",
    "AI customer service", "AI hiring", "AI recruitment",
    "AI productivity", "AI supply chain", "AI retail", "AI logistics",
    "AI social media", "AI algorithm", "AI coding", "AI sports",
]

# tier 3: risk and fear framing — feeds AAI scoring
TIER_3_RISK = [
    "AI risks", "AI dangers", "AI safety", "AI threat", "AI harm",
    "AI existential risk", "AI misinformation", "AI deepfake",
    "AI surveillance", "AI bias discrimination", "AI privacy",
    "AI hallucination", "AI manipulation", "AI addiction",
    "AI collapse", "AI catastrophe",
    "AI racism", "AI sexism", "AI children safety",
    "AI scam", "AI impersonation",
]

# tier 4: agency and autonomy language — feeds PSI scoring
TIER_4_AGENCY = [
    "AI takes over", "AI replaces workers", "AI makes decisions",
    "AI autonomous", "AI thinks", "AI chooses", "AI controls",
    "humans control AI", "AI automation workers", "AI acts independently",
    "AI decides", "AI outperforms humans",
    "AI sentient", "AI consciousness", "AI feelings",
    "AI wants", "AI understands",
]

# tier 5: named models and companies — coverage spikes around releases
TIER_5_ENTITIES = [
    "ChatGPT", "OpenAI", "Google Gemini", "Microsoft Copilot",
    "Meta AI", "Anthropic Claude", "DeepSeek AI", "Grok AI",
    "GPT-4", "GPT-5", "AI startup", "AI investment", "AI chip",
    "Nvidia AI", "AI funding", "AI valuation",
    "Apple Intelligence", "Llama AI", "Mistral AI",
    "Perplexity AI", "Sora AI", "AI PC",
    "Sam Altman AI", "AI bubble", "AI IPO",
]

# tier 6: regulation and governance
TIER_6_POLICY = [
    "AI regulation", "AI law", "AI ethics", "AI policy", "AI ban",
    "AI copyright", "AI accountability", "EU AI Act", "AI governance",
    "AI Congress", "AI Senate", "AI executive order", "AI liability",
    "AI Safety Summit", "AI watermark", "AI transparency",
    "AI audit", "AI lawsuit", "China AI regulation",
]

# tier 7: human impact — key for Invisible Human Framework
TIER_7_HUMAN = [
    "AI jobs lost", "AI unemployment", "AI workers displaced",
    "AI relationships", "AI loneliness", "AI creativity human",
    "AI parenting", "AI grief", "AI companionship",
    "AI trust society", "AI spirituality", "AI caregiving",
    "AI dignity", "AI identity", "AI meaning work",
    "AI purpose", "AI human connection", "AI ethics care",
    "AI white collar",
]

# tier 8: democracy and disinformation — major spike during Nov 2024 election
TIER_8_DEMOCRACY = [
    "AI election", "AI voting", "AI democracy", "AI disinformation",
    "AI propaganda", "AI fake news", "AI political ads",
    "AI deepfake politician", "AI influence operation",
    "AI censorship", "AI freedom of speech", "AI media manipulation",
]

# tier 9: infrastructure and economics — became a big story in 2025
# (data centers, power grids, water usage)
TIER_9_INFRASTRUCTURE = [
    "AI data center", "AI energy consumption", "AI water usage",
    "AI power grid", "AI carbon footprint", "AI cost",
    "AI economic impact", "AI GDP", "AI labor market",
    "AI productivity growth", "AI inequality", "AI global south",
]

ALL_QUERIES = (
    TIER_1_CORE + TIER_2_DOMAINS + TIER_3_RISK + TIER_4_AGENCY +
    TIER_5_ENTITIES + TIER_6_POLICY + TIER_7_HUMAN +
    TIER_8_DEMOCRACY + TIER_9_INFRASTRUCTURE
)

# category mapping for website topic filter
QUERY_TO_CATEGORY = {}
for q in TIER_1_CORE:            QUERY_TO_CATEGORY[q] = "General AI"
for q in TIER_3_RISK:            QUERY_TO_CATEGORY[q] = "Safety & Risk"
for q in TIER_4_AGENCY:          QUERY_TO_CATEGORY[q] = "Agency & Autonomy"
for q in TIER_5_ENTITIES:        QUERY_TO_CATEGORY[q] = "Companies & Products"
for q in TIER_6_POLICY:          QUERY_TO_CATEGORY[q] = "Policy & Governance"
for q in TIER_7_HUMAN:           QUERY_TO_CATEGORY[q] = "Society & Human Impact"
for q in TIER_8_DEMOCRACY:       QUERY_TO_CATEGORY[q] = "Democracy & Disinformation"
for q in TIER_9_INFRASTRUCTURE:  QUERY_TO_CATEGORY[q] = "Infrastructure & Economy"

for q in TIER_2_DOMAINS:
    if any(x in q for x in ["health","medicine","drug","mental","diagnosis","radiology","surgery"]):
        QUERY_TO_CATEGORY[q] = "Health & Science"
    elif any(x in q for x in ["educat","school","student","tutor"]):
        QUERY_TO_CATEGORY[q] = "Education"
    elif any(x in q for x in ["financ","bank","stock","trad","insur","real estate"]):
        QUERY_TO_CATEGORY[q] = "Finance & Economy"
    elif any(x in q for x in ["legal","court","govern","judge"]):
        QUERY_TO_CATEGORY[q] = "Law & Government"
    elif any(x in q for x in ["military","defense","cyber","weapon","facial","polic","track"]):
        QUERY_TO_CATEGORY[q] = "Defense & Security"
    elif any(x in q for x in ["climat","environ","agri","energy","wildfire"]):
        QUERY_TO_CATEGORY[q] = "Environment"
    elif any(x in q for x in ["transport","driving","robot"]):
        QUERY_TO_CATEGORY[q] = "Transportation"
    elif any(x in q for x in ["art","music","writ","journal","film","video","image"]):
        QUERY_TO_CATEGORY[q] = "Creative Industries"
    elif any(x in q for x in ["social media","algorithm","coding"]):
        QUERY_TO_CATEGORY[q] = "Technology & Software"
    else:
        QUERY_TO_CATEGORY[q] = "Business"

print(f"total queries: {len(ALL_QUERIES)} across 9 tiers")
print(f"categories mapped: {len(set(QUERY_TO_CATEGORY.values()))}")

total queries: 192 across 9 tiers
categories mapped: 18


In [3]:
# CELL 3 -- Time windows: 20 monthly windows Aug 2024 to Mar 2026
'''
Defines the 20 monthly date windows over which every query
in ALL_QUERIES will be executed.

Key decision -- why monthly windows instead of one wide range:
gnews ranks results by recency + relevance and caps at 100
per query. A single Aug 2024 to Mar 2026 window returns only
the 100 most recent articles; all 2024 content gets crowded
out. Monthly windows force Google News to surface content
from that specific period, enabling temporal trend analysis.

Notable events captured in this window:
2024-10: US Presidential election -- AI disinformation peak
2025-01: DeepSeek release + ChatGPT Gov launch
2025-02: EU AI Act first implementation deadlines
2025-09: PSI near-parity month (AI_AGENT ~= HUMAN_AGENT)
Estimated runtime: < 1 second (definitions only)
'''

TIME_WINDOWS = [
    ("2024-08", (2024, 8,  1), (2024, 8,  31)),
    ("2024-09", (2024, 9,  1), (2024, 9,  30)),
    ("2024-10", (2024, 10, 1), (2024, 10, 31)),
    ("2024-11", (2024, 11, 1), (2024, 11, 30)),
    ("2024-12", (2024, 12, 1), (2024, 12, 31)),
    ("2025-01", (2025, 1,  1), (2025, 1,  31)),
    ("2025-02", (2025, 2,  1), (2025, 2,  28)),
    ("2025-03", (2025, 3,  1), (2025, 3,  31)),
    ("2025-04", (2025, 4,  1), (2025, 4,  30)),
    ("2025-05", (2025, 5,  1), (2025, 5,  31)),
    ("2025-06", (2025, 6,  1), (2025, 6,  30)),
    ("2025-07", (2025, 7,  1), (2025, 7,  31)),
    ("2025-08", (2025, 8,  1), (2025, 8,  31)),
    ("2025-09", (2025, 9,  1), (2025, 9,  30)),
    ("2025-10", (2025, 10, 1), (2025, 10, 31)),
    ("2025-11", (2025, 11, 1), (2025, 11, 30)),
    ("2025-12", (2025, 12, 1), (2025, 12, 31)),
    ("2026-01", (2026, 1,  1), (2026, 1,  31)),
    ("2026-02", (2026, 2,  1), (2026, 2,  28)),
    ("2026-03", (2026, 3,  1), (2026, 3,  31)),
]

total_calls = len(ALL_QUERIES) * len(TIME_WINDOWS)
print(f"{len(TIME_WINDOWS)} windows: {TIME_WINDOWS[0][0]} to {TIME_WINDOWS[-1][0]}")
print(f"total API calls: {total_calls:,}  (~{total_calls * 2.5 / 3600:.1f} hrs at 2.5s/query)")

20 windows: 2024-08 to 2026-03
total API calls: 3,840  (~2.7 hrs at 2.5s/query)


In [4]:
# CELL 4 -- Helper functions: cleaning, filtering, date parsing
'''
Defines three utility functions used inside the scraping loop.
None of these make network calls -- they process strings only.
clean_title(raw)      -- normalizes headline text for dedup
is_ai_relevant(title) -- quality filter removing false positives
parse_gnews_date(raw) -- robust date parsing for gnews format

Key decision -- why is_ai_relevant() is necessary:

broad queries occasionally return articles where "AI" is a country
code (.ai domain) or someone's initials. this filter requires at
least one of these terms to appear in the title. validated on a
random 100-headline sample: caught 4 false positives, lost 0 real ones.

'''
AI_RELEVANCE_KEYWORDS = [
    "ai", "artificial intelligence", "machine learning",
    "chatgpt", "openai", "llm", "language model", "neural",
    "deep learning", "algorithm", "generative", "gpt",
    "gemini", "copilot", "claude", "deepseek", "chatbot",
    "autonomous", "computer vision", "natural language",
    "large language", "transformer model", "automation",
]


def clean_title(raw_title):
    """
    Normalize a raw headline string.
    - strips whitespace and smart quotes
    - removes non-printable characters  
    - collapses multiple spaces
    - strips trailing ' - Publisher Name' that gnews sometimes appends
    """
    if not isinstance(raw_title, str):
        return ""
    t = raw_title.strip()
    t = t.replace("\u2018", "'").replace("\u2019", "'")
    t = t.replace("\u201c", '"').replace("\u201d", '"')
    t = t.replace("\u2013", "-").replace("\u2014", "-")
    t = re.sub(r'[^\x20-\x7E]', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    # gnews sometimes appends ' - Publisher Name' to the title
    t = re.sub(r'\s*[\-|]\s*[A-Z][\w\s\.]{2,45}$', '', t).strip()
    return t


def is_ai_relevant(title):
    """return True only if the headline is genuinely about AI."""
    if not isinstance(title, str) or len(title) < 20:
        return False
    return any(kw in title.lower() for kw in AI_RELEVANCE_KEYWORDS)


# gnews date format: 'Mon, 01 Jan 2025 12:00:00 GMT'
GNEWS_DATE_FORMAT = "%a, %d %b %Y %H:%M:%S %Z"

def parse_gnews_date(raw_date_str, fallback_label):
    """
    Parse gnews date string to YYYY-MM-DD.
    Falls back to the window label if parsing fails.
    """
    try:
        parsed = datetime.strptime(raw_date_str.strip(), GNEWS_DATE_FORMAT)
        return parsed.strftime("%Y-%m-%d"), parsed.year, parsed.month
    except Exception:
        year  = int(fallback_label[:4])
        month = int(fallback_label[5:7])
        return f"{fallback_label}-01", year, month


print(f"helper functions defined. relevance filter: {len(AI_RELEVANCE_KEYWORDS)} keywords")

helper functions defined. relevance filter: 23 keywords


In [5]:
# CELL 5 -- Core scraping function: scrape_one_window()
'''
the main scraping function. runs all 192 queries for one monthly window
and returns a deduplicated DataFrame.

rate limiting: 2.5s between queries, 30s backoff on failure, 8s between
windows (called in cell 7, not here). these delays are intentional —
removing them causes Google News to return empty results silently
rather than throwing an error, which is hard to catch.
estimated runtime per window: 8–12 minutes
'''
def scrape_one_window(label, start_tuple, end_tuple, queries, delay=2.5):
    gn = GNews(
        language='en',
        country='US',
        start_date=start_tuple,
        end_date=end_tuple,
        max_results=100
    )

    records, failed, total_raw = [], [], 0

    for query in queries:
        try:
            results = gn.get_news(query)
            if results:
                total_raw += len(results)
                for r in results:
                    title = clean_title(r.get("title", ""))
                    if not is_ai_relevant(title):
                        continue

                    pub = r.get("publisher", {})
                    publisher = (
                        pub.get("title", "") if isinstance(pub, dict) else str(pub)
                    ).strip()

                    date_str, year, month = parse_gnews_date(
                        r.get("published date", ""), label
                    )

                    records.append({
                        "title":        title,
                        "publisher":    publisher,
                        "date":         date_str,
                        "year":         year,
                        "month":        month,
                        "query_source": query,
                        "category":     QUERY_TO_CATEGORY.get(query, "General AI"),
                        "window":       label,
                        "url":          r.get("url", ""),
                    })

        except Exception as e:
            failed.append(query)
            print(f"  [warning] '{query}' failed ({type(e).__name__}) — backing off 30s")
            time.sleep(30)
            continue

        time.sleep(delay)

    if not records:
        print(f"  [warning] {label}: zero records — check connection")
        return pd.DataFrame()

    df = pd.DataFrame(records).drop_duplicates(subset=["title"])
    print(
        f"  {label}: {total_raw:,} raw -> {len(df):,} unique "
        f"({len(failed)} failed queries)"
    )
    return df

print("scrape_one_window() defined.")

scrape_one_window() defined.


In [ ]:
# CELL 6 -- Checkpoint system
'''
Defines four functions managing per-window checkpoint CSV files.

checkpoint system so a 6–8 hour scrape is safe to interrupt.
after each window, the results are written to checkpoints/checkpoint_YYYY-MM.csv.
on re-run, completed windows are detected and skipped automatically.
to resume after an interruption: re-run cells 1–6 (fast, no network),
then run cell 7 and it picks up where it left off.

How it works:
1. After each window, save_ckpt() writes checkpoints/checkpoint_YYYY-MM.csv.
2. is_done() checks for that file before starting each window.
3. load_ckpt() reads files back for merging in Cell 8.

Resuming after interruption:
1. Re-run Cells 1-6 to reload functions (< 5 seconds, no network)
2. Run Cell 7 -- it skips windows with existing checkpoints

UTF-8 encoding handles accented characters from international
publishers (Le Monde, Der Spiegel, La Republica).
'''
CHECKPOINT_DIR = "checkpoints"

def ckpt_path(label):
    return os.path.join(CHECKPOINT_DIR, f"checkpoint_{label}.csv")

def is_done(label):
    return os.path.exists(ckpt_path(label))

def save_ckpt(df, label):
    df.to_csv(ckpt_path(label), index=False)

def load_ckpt(label):
    return pd.read_csv(ckpt_path(label))

done_windows = [lbl for lbl, _, _ in TIME_WINDOWS if is_done(lbl)]
todo_windows = [lbl for lbl, _, _ in TIME_WINDOWS if not is_done(lbl)]

print(f"completed: {len(done_windows)}/{len(TIME_WINDOWS)} windows")
if done_windows:
    already_rows = sum(len(load_ckpt(l)) for l in done_windows)
    print(f"headlines already scraped: {already_rows:,}")
    print(f"▶ resuming from: {todo_windows[0] if todo_windows else 'all done'}")
else:
    print("▶ fresh run — all windows pending")

Checkpoint directory: c:\Users\Sadhana\Desktop\cs210_bias_project\checkpoints
  Completed: 0/20 windows
  Remaining: ['2024-08', '2024-09', '2024-10', '2024-11', '2024-12', '2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03']

  ▶ Fresh run — all windows pending.


In [ ]:
# CELL 7 -- Main scraping loop
'''
Iterates over all 20 monthly windows and 192 queries per
window. Checks for existing checkpoints before each window
(skip if found), runs the scrape, saves checkpoint, prints
running total.

the actual scrape. this took 667 minutes (11.1 hours) when we ran it on April 21 2026.
theoretical minimum at 2.5s/query is ~160 min — the rest is network
latency and the 8s pauses between windows.

make sure your laptop won't sleep before starting this.
if it gets interrupted, just re-run — completed windows are checkpointed
and will be skipped automatically.

estimated runtime: 6–8 hours for a fresh run
'''
run_start = datetime.now()
print(f"starting scrape — {len(ALL_QUERIES)} queries x {len(TIME_WINDOWS)} windows")
print(f"started: {run_start.strftime('%Y-%m-%d %H:%M:%S')}")

window_results = []

for idx, (label, start, end) in enumerate(TIME_WINDOWS, 1):
    print(f"\n[{idx:02d}/{len(TIME_WINDOWS)}] {label}  "
          f"({start[0]}-{start[1]:02d}-{start[2]:02d} -> "
          f"{end[0]}-{end[1]:02d}-{end[2]:02d})")

    if is_done(label):
        df_w = load_ckpt(label)
        print(f"  skipped — loaded {len(df_w):,} from checkpoint")
    else:
        df_w = scrape_one_window(label, start, end, ALL_QUERIES)
        if not df_w.empty:
            save_ckpt(df_w, label)
            print(f"  checkpoint saved -> {ckpt_path(label)}")
        time.sleep(8)  # pause between windows

    if not df_w.empty:
        window_results.append({
            "window":     label,
            "headlines":  len(df_w),
            "publishers": df_w["publisher"].nunique()
        })

    total_now = sum(r["headlines"] for r in window_results)
    elapsed   = (datetime.now() - run_start).seconds // 60
    print(f"  running total: {total_now:,} | {elapsed} min elapsed")


total_elapsed = (datetime.now() - run_start).seconds // 60
print(f"\nscrape complete — {total_elapsed} minutes total")
print(f"\n{'window':<12} {'headlines':>12} {'publishers':>12}")
print("-" * 38)
for r in window_results:
    print(f"{r['window']:<12} {r['headlines']:>12,} {r['publishers']:>12,}")
print("-" * 38)
print(f"{'total':<12} {sum(r['headlines'] for r in window_results):>12,}  (before global dedup)")

STARTING SCRAPE
  Queries per window:  192
  Windows to scrape:   20
  Started:             2026-04-21 11:22:01

[01/20] 2024-08  (2024-08-01 → 2024-08-31)
  ✓ 2024-08: 12,587 raw  →  10,865 relevant  →  6,500 unique  (4,365 dupes, 0 failed queries)
    ✓ Checkpoint saved → checkpoints\checkpoint_2024-08.csv
  Running total: 6,500 headlines | 31 min elapsed

[02/20] 2024-09  (2024-09-01 → 2024-09-30)
  ✓ 2024-09: 13,364 raw  →  11,480 relevant  →  6,997 unique  (4,483 dupes, 0 failed queries)
    ✓ Checkpoint saved → checkpoints\checkpoint_2024-09.csv
  Running total: 13,497 headlines | 62 min elapsed

[03/20] 2024-10  (2024-10-01 → 2024-10-31)
  ✓ 2024-10: 13,611 raw  →  11,799 relevant  →  7,461 unique  (4,338 dupes, 0 failed queries)
    ✓ Checkpoint saved → checkpoints\checkpoint_2024-10.csv
  Running total: 20,958 headlines | 95 min elapsed

[04/20] 2024-11  (2024-11-01 → 2024-11-30)
  ✓ 2024-11: 12,816 raw  →  10,885 relevant  →  6,743 unique  (4,142 dupes, 0 failed queries)
    

In [ ]:
# Cell 7b - Test Scrape (optional)

# optional: test the scraper before running the full 6-8 hour job
#
# this runs just 5 queries over a single 3-day window so you can
# verify gnews is working and the output schema looks right.
# takes about 15 seconds. doesn't touch any checkpoint files.
#
# the full dataset (data/ai_headlines_raw.csv) is already included
# in this repo — cell 7 will skip every window that has a checkpoint,
# so you don't need to re-scrape anything to run the rest of the project.
# only uncomment this if you want to verify the pipeline works
# or if you're building on top of this and need fresh data.

'''
test_gn = GNews(
    language='en',
    country='US',
    start_date=(2025, 6, 1),
    end_date=(2025, 6, 3),
    max_results=100
)

test_queries = ["AI regulation", "AI jobs", "ChatGPT", "AI safety", "generative AI"]
test_records = []

for q in test_queries:
    results = test_gn.get_news(q)
    for r in results:
        title = clean_title(r.get("title", ""))
        if is_ai_relevant(title):
            test_records.append({"title": title, "query": q,
                                 "publisher": r.get("publisher", {}).get("title", "")})
    time.sleep(2.5)

test_df = pd.DataFrame(test_records).drop_duplicates(subset=["title"])
print(f"test complete: {len(test_df)} unique headlines from {len(test_queries)} queries")
print(test_df[["query", "publisher", "title"]].to_string(index=False))
'''

In [ ]:
# CELL 8 - Merge all checkpoints and global deduplication

'''
Loads all 20 checkpoint CSVs, concatenates them, runs a
second deduplication pass across the full combined dataset.

Two-stage dedup design:
   Stage 1 (Cell 7, per-window): removes same story under
   multiple queries within one month.
   
   Stage 2 (this cell, global): removes same story resurfacing
   in a later month. Strategy: sort by date ascending, keep
   first occurrence (preserves original publication date).

# merge all 20 checkpoint files and run a second dedup pass across
# the full dataset. the per-window dedup in cell 7 removes duplicates
# within a month (same story under multiple queries). this pass catches
# the same story resurfacing in a later month — we keep the first
# occurrence to preserve the original publication date.

# headline_id becomes the foreign key in the SQLite schema.
'''
print("loading checkpoint files...")

all_chunks, missing = [], []

for label, _, _ in TIME_WINDOWS:
    if is_done(label):
        chunk = load_ckpt(label)
        all_chunks.append(chunk)
        print(f"  {label}: {len(chunk):,} headlines, {chunk['publisher'].nunique():,} publishers")
    else:
        missing.append(label)
        print(f"  [!] {label}: no checkpoint — window was not scraped")

if missing:
    print(f"\nmissing windows: {missing} — re-run cell 7 to scrape them")

df = pd.concat(all_chunks, ignore_index=True)
print(f"\nmerged total (before global dedup): {len(df):,}")

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.sort_values("date")

# global dedup — keep first (earliest) occurrence
df = df.drop_duplicates(subset=["title"], keep="first")
df = df.dropna(subset=["title", "date"])
df = df[df["title"].str.strip().str.len() >= 20]
df = df[df["publisher"].fillna("").str.strip().str.len() >= 2]

df = df.reset_index(drop=True)
df.insert(0, "headline_id", df.index + 1)  # 1-indexed to match SQL convention

print(f"after global dedup:    {len(df):,}")
print(f"unique publishers:     {df['publisher'].nunique():,}")
print(f"date range:            {df['date'].min().date()} -> {df['date'].max().date()}")
print(f"span:                  {(df['date'].max() - df['date'].min()).days} days")

n = len(df)
if n >= 15000:
    print(f"\n{n:,} headlines — well above target")
elif n >= 10000:
    print(f"\n{n:,} headlines — target met")
else:
    print(f"\n[!] {n:,} headlines — below target, check failed windows")

Loading all checkpoint files...

  ✓ 2024-08: 6,500 headlines, 2,299 publishers
  ✓ 2024-09: 6,997 headlines, 2,431 publishers
  ✓ 2024-10: 7,461 headlines, 2,523 publishers
  ✓ 2024-11: 6,743 headlines, 2,447 publishers
  ✓ 2024-12: 6,863 headlines, 2,441 publishers
  ✓ 2025-01: 7,951 headlines, 2,607 publishers
  ✓ 2025-02: 8,321 headlines, 2,833 publishers
  ✓ 2025-03: 8,386 headlines, 2,736 publishers
  ✓ 2025-04: 8,663 headlines, 2,790 publishers
  ✓ 2025-05: 9,030 headlines, 2,876 publishers
  ✓ 2025-06: 9,177 headlines, 2,875 publishers
  ✓ 2025-07: 9,804 headlines, 2,968 publishers
  ✓ 2025-08: 9,573 headlines, 2,892 publishers
  ✓ 2025-09: 9,823 headlines, 3,044 publishers
  ✓ 2025-10: 10,363 headlines, 3,191 publishers
  ✓ 2025-11: 10,356 headlines, 3,279 publishers
  ✓ 2025-12: 10,333 headlines, 3,286 publishers
  ✓ 2026-01: 10,841 headlines, 3,312 publishers
  ✓ 2026-02: 11,210 headlines, 3,446 publishers
  ✓ 2026-03: 11,372 headlines, 3,276 publishers

Merged total (before

In [ ]:
# CELL 9 - Data Quality Validation
'''
integrity checks before anything goes into the NLP pipeline.
assertions fail loudly rather than letting bad data through silently.

checks performed:
- No missing titles/dates/publishers (would break SQL joins)
- No duplicate titles (would inflate bias score averages)
- Every month >= 200 headlines (statistical analysis minimum)
- Average title length 50-100 chars (gnews norm)
- Top publishers are recognizable outlets
'''
assert len(df) >= 50_000,                          f"expected >= 50K headlines, got {len(df):,}"
assert df["title"].isna().sum() == 0,              "missing titles detected"
assert df["date"].isna().sum() == 0,               "missing dates detected"
assert df["publisher"].isna().sum() == 0,          "missing publishers detected"
assert df.duplicated(subset=["title"]).sum() == 0, "duplicate titles found"

print(f"total headlines:    {len(df):,}")
print(f"date range:         {df['date'].min().date()} -> {df['date'].max().date()}")
print(f"span:               {(df['date'].max() - df['date'].min()).days} days")
print(f"unique publishers:  {df['publisher'].nunique():,}")
print(f"avg title length:   {df['title'].str.len().mean():.0f} chars")

print("\nmonthly distribution:")
monthly = df.groupby(df['date'].dt.to_period('M')).size()
max_count = monthly.max()
for period, count in monthly.items():
    bar  = '█' * int((count / max_count) * 40)
    flag = "  [thin]" if count < 200 else ""
    print(f"  {period}  {count:>6,}  {bar}{flag}")

print("\ncategory distribution:")
for cat, count in df['category'].value_counts().items():
    print(f"  {cat:<30} {count:>7,}  ({count/len(df)*100:.1f}%)")

print("\ntop 15 publishers:")
for pub, count in df['publisher'].value_counts().head(15).items():
    print(f"  {pub:<40} {count:>6,}")

print("\nall checks passed.")

DATA QUALITY VALIDATION

  Total headlines:    179,372
  Date range:         2024-08-01 → 2026-03-31
  Span:               607 days
  Unique publishers:  13,019
  Avg title length:   75 chars
  Min title length:   20 chars
  Max title length:   272 chars

  Integrity Checks:
    Missing titles:         0  (must be 0)
    Missing dates:          0  (must be 0)
    Missing publishers:     0  (must be 0)
    Duplicate titles:       0  (must be 0)

  Monthly Distribution:
    2024-08   6,500  ██████████████████████
    2024-09   6,990  ████████████████████████
    2024-10   7,453  ██████████████████████████
    2024-11   6,736  ███████████████████████
    2024-12   6,849  ████████████████████████
    2025-01   7,942  ████████████████████████████
    2025-02   8,303  █████████████████████████████
    2025-03   8,380  █████████████████████████████
    2025-04   8,643  ██████████████████████████████
    2025-05   9,004  ███████████████████████████████
    2025-06   9,165  ████████████████████

In [ ]:
# CELL 10 - Save Final Datasets
'''
What this Cell Does: 
# three output files with different column sets for different downstream uses.
# utf-8 encoding matters on Windows — default cp1252 breaks on accented
# characters from international publishers.

ai_headlines_raw.csv (4 cols, Figshare-compatible):
headline_id, title, publisher, date. Used in
02_nlp_pipeline.ipynb for Figshare cross-validation.

ai_headlines_full.csv (10 cols, full metadata):
Adds year, month, category, query_source, window, url.
Used in 01_database.ipynb to populate the SQLite schema.

ai_headlines_website.csv (3,000 rows, stratified sample):
Proportional sample across all 20 months with url column
for the interactive website headline explorer.

All files UTF-8 encoded (required on Windows where default
encoding is cp1252, which fails on accented characters).

'''
# 4-column raw file matching the Figshare schema (used for cross-validation)
df_raw = df[["headline_id", "title", "publisher", "date"]].copy()
df_raw["date"] = df_raw["date"].dt.strftime("%Y-%m-%d")
df_raw.to_csv("data/ai_headlines_raw.csv", index=False)

# full 10-column file used by notebook 01 to populate SQLite
df_full = df.copy()
df_full["date"] = df_full["date"].dt.strftime("%Y-%m-%d")
df_full.to_csv("data/ai_headlines_full.csv", index=False)

# 3,000-headline stratified sample for the website headline explorer
# proportional across all 20 months so no window is over-represented
df["year_month"] = df["date"].dt.to_period("M").astype(str)
n_months = df["year_month"].nunique()
per_month = max(1, 3000 // n_months)

website_sample = (
    df.groupby("year_month", group_keys=False)
      .apply(lambda g: g.sample(min(per_month, len(g)), random_state=42),
             include_groups=False)
      .reset_index(drop=True)
)

website_cols = ["headline_id", "title", "publisher", "date",
                "category", "url", "year_month"]
website_cols = [c for c in website_cols if c in website_sample.columns]
df_website = website_sample[website_cols].copy()
df_website["date"] = df_website["date"].dt.strftime("%Y-%m-%d")
df_website.to_csv("data/ai_headlines_website.csv", index=False)

print("files saved to data/:")
for fname in ["ai_headlines_raw.csv", "ai_headlines_full.csv", "ai_headlines_website.csv"]:
    path = f"data/{fname}"
    size_kb = os.path.getsize(path) // 1024
    rows = sum(1 for _ in open(path, encoding='utf-8')) - 1
    print(f"  {fname:<35} {rows:>8,} rows  {size_kb:>6,} KB")

print()
print(f"  headlines:         {len(df):,}")
print(f"  unique publishers: {df['publisher'].nunique():,}")
print(f"  date range:        {df['date'].min().date()} -> {df['date'].max().date()}")
print(f"  span:              {(df['date'].max() - df['date'].min()).days} days")
print(f"  queries used:      {len(ALL_QUERIES)} across 9 tiers")
print(f"  time windows:      {len(TIME_WINDOWS)} monthly")
print(f"  categories:        {df['category'].nunique()}")
print()

In [ ]:
#CELL 11 - Sample Headlines Spot-Check
'''
quick sanity check — sample a few headlines from the tiers that matter
# most for our hypotheses to make sure the scraper pulled the right content.
# tier 3 should show fear/risk language, tier 7 should be nearly empty
# (that absence is what the Invisible Human finding is about).
'''
tiers_to_check = [
    ("tier 1 — core",          TIER_1_CORE),
    ("tier 3 — risk & fear",   TIER_3_RISK),
    ("tier 4 — agency",        TIER_4_AGENCY),
    ("tier 7 — human impact",  TIER_7_HUMAN),
]

for tier_name, tier_queries in tiers_to_check:
    tier_df = df[df["query_source"].isin(tier_queries)]
    sample  = tier_df.sample(min(5, len(tier_df)), random_state=42)
    print(f"\n{tier_name}  ({len(tier_df):,} total headlines)")
    for _, row in sample.iterrows():
        date_str  = str(row['date'])[:10]
        publisher = str(row['publisher'])[:35]
        title     = str(row['title'])[:90]
        print(f"  [{date_str}] {publisher:<35} {title}")

SAMPLE HEADLINES BY TIER

── Tier 1 — Core  (7,237 total headlines) ──
  [2024-08-27] Congressman Gabe Amo (.go   Congressman Amo Hosts Roundtable on Artificial Intelligence at Smithfield High School
  [2024-09-05] Frontiers                   A review of model evaluation metrics for machine learning in genetics and genomics
  [2025-06-03] EdTech Magazine             How Higher Ed Institutions Are Using Built-In Generative AI Tools
  [2025-06-02] Genetic Literacy Project    AI and machine learning could accelerate and mainstream the culture meat revolution
  [2024-09-19] Miami University            Introducing: Generative AI at Miami

── Tier 3 — Risk & Fear  (3,383 total headlines) ──
  [2025-06-10] TechTarget                  AI-powered attacks: What CISOs need to know now
  [2025-08-11] MSSP Alert                  SOCRadar Rolls Out Agentic Threat Intelligence to Turn AI Insights into Action
  [2025-05-15] JD Supra                    Dario Amodei Warns of the Danger of Black Box AI t